# TensorRT Edge Inference Benchmark — Colab T4

Self-contained benchmark notebook for evaluating YOLOv8n under TensorRT FP32 / FP16 / INT8 precision constraints on a Colab T4 GPU.

**Prerequisites before running:**
- Set runtime to T4 GPU: `Runtime → Change runtime type → T4 GPU`
- Upload `yolov8n.onnx` to Google Drive at `edge-inference-benchmark/yolov8n.onnx`
- Upload `data/calibration/` (500 JPEG images + manifest.json) to Drive at `edge-inference-benchmark/calibration/`
- Upload `data/val2017/` and `data/annotations/` to Drive for mAP evaluation

**Each cell is idempotent** — re-running any cell will not corrupt state.

In [ ]:
# ── Cell 1 ── Environment Setup ───────────────────────────────────────────────
# Install pinned packages, verify CUDA and TensorRT are available on this T4 instance.
# Document runtime environment for reproducibility — required per benchmark protocol.
# Idempotent: pip skips already-installed versions.

import subprocess
import sys

# GPU and CUDA check — must be T4 with CUDA 11.8
!nvidia-smi

# Install pinned versions — see requirements-colab.txt
!pip install -q \
    onnx==1.16.0 \
    onnxruntime-gpu==1.18.0 \
    pycocotools==2.0.7 \
    numpy==1.26.4 \
    opencv-python-headless==4.10.0.84

# torch, TensorRT, pycuda come pre-installed on Colab T4
import torch
import tensorrt as trt
import onnxruntime as ort
import numpy as np

colab_env = {
    "python": sys.version.split()[0],
    "cuda": torch.version.cuda,
    "torch": torch.__version__,
    "tensorrt": trt.__version__,
    "onnxruntime": ort.__version__,
    "numpy": np.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    if torch.cuda.is_available() else 0,
}

print("\n=== Colab Runtime Environment ===")
for k, v in colab_env.items():
    print(f"  {k:20s}: {v}")

assert torch.cuda.is_available(), (
    "GPU not available — switch to T4: Runtime → Change runtime type → T4 GPU"
)
print("\nEnvironment check passed.")

In [ ]:
# ── Cell 2 ── Mount Drive / Clone Repo / Load ONNX Model ─────────────────────
# Mount Google Drive to access the exported ONNX model and calibration data.
# Clone or update the benchmark repo so src/ modules are importable.
# Idempotent: git pull is safe to re-run; model copy is guarded by exists check.

import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = "https://github.com/KNakul242/edge-inference-benchmark.git"
REPO_DIR = Path("/content/edge-inference-benchmark")
DRIVE_BASE = Path("/content/drive/MyDrive/edge-inference-benchmark")

# Clone or update repo
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# Copy ONNX model from Drive
LOCAL_MODEL = REPO_DIR / "models/yolov8n.onnx"
LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)

if not LOCAL_MODEL.exists():
    drive_model = DRIVE_BASE / "yolov8n.onnx"
    assert drive_model.exists(), (
        f"ONNX model not found at {drive_model}.\n"
        "Export it locally: python scripts/export_model.py\n"
        "Then upload to Drive at: edge-inference-benchmark/yolov8n.onnx"
    )
    shutil.copy(drive_model, LOCAL_MODEL)
    print(f"Model copied from Drive → {LOCAL_MODEL}")
else:
    print(f"Model already present: {LOCAL_MODEL}")

print(f"ONNX model size: {LOCAL_MODEL.stat().st_size / 1e6:.1f} MB")

# Sanity-check ONNX model output shape — must be (1, 84, 8400) for YOLOv8n
session = ort.InferenceSession(str(LOCAL_MODEL), providers=["CUDAExecutionProvider"])
input_name = session.get_inputs()[0].name
dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
out = session.run(None, {input_name: dummy})
assert out[0].shape == (1, 84, 8400), f"Unexpected output shape: {out[0].shape}"
del session
print(f"ONNX sanity check passed — output shape: {out[0].shape}")

ENGINE_DIR = REPO_DIR / "models"
ENGINE_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Cell 3 ── TensorRT FP32 Engine Conversion ─────────────────────────────────
# Build a TensorRT FP32 engine from the ONNX model and serialize to disk.
# Idempotent: skips build if engine file already exists.
# Build time: approximately 1-2 minutes on T4.

import tensorrt as trt


def build_trt_engine(
    onnx_path: str,
    engine_path: str,
    precision: str = "fp32",
    calibrator=None,
) -> str:
    """Build and serialize a TensorRT engine from an ONNX model.

    Args:
        onnx_path: Path to the source .onnx file (opset 17, static shape).
        engine_path: Destination path for the serialized .engine file.
        precision: One of 'fp32', 'fp16', 'int8'.
        calibrator: IInt8EntropyCalibrator2 instance (required for int8 only).

    Returns:
        Absolute path to the written engine file.
    """
    trt_logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(trt_logger)
    network = builder.create_network(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    parser = trt.OnnxParser(network, trt_logger)

    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
            raise RuntimeError("ONNX parse failed:\n" + "\n".join(errors))

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1 GB

    if precision == "fp16":
        assert builder.platform_has_fast_fp16, "FP16 not supported on this GPU"
        config.set_flag(trt.BuilderFlag.FP16)
    elif precision == "int8":
        assert builder.platform_has_fast_int8, "INT8 not supported on this GPU"
        config.set_flag(trt.BuilderFlag.INT8)
        assert calibrator is not None, "Calibrator required for INT8 precision"
        config.int8_calibrator = calibrator

    print(f"Building TensorRT {precision.upper()} engine — this may take 1-3 minutes...")
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError(f"TensorRT engine build failed (precision={precision})")

    with open(engine_path, "wb") as f:
        f.write(engine_bytes)

    size_mb = os.path.getsize(engine_path) / 1e6
    print(f"Engine saved: {engine_path} ({size_mb:.1f} MB)")
    return str(engine_path)


FP32_ENGINE = ENGINE_DIR / "yolov8n_fp32.engine"

if not FP32_ENGINE.exists():
    build_trt_engine(str(LOCAL_MODEL), str(FP32_ENGINE), precision="fp32")
else:
    print(f"FP32 engine already exists ({FP32_ENGINE.stat().st_size / 1e6:.1f} MB), skipping build.")

In [ ]:
# ── Cell 4 ── TensorRT FP16 Engine Conversion ─────────────────────────────────
# Build a TensorRT FP16 engine. Colab T4 has native FP16 tensor cores.
# Idempotent: skips build if engine already exists.
# Build time: approximately 1-2 minutes.

FP16_ENGINE = ENGINE_DIR / "yolov8n_fp16.engine"

if not FP16_ENGINE.exists():
    build_trt_engine(str(LOCAL_MODEL), str(FP16_ENGINE), precision="fp16")
else:
    print(f"FP16 engine already exists ({FP16_ENGINE.stat().st_size / 1e6:.1f} MB), skipping build.")

In [ ]:
# ── Cell 5 ── TensorRT INT8 Engine Conversion ─────────────────────────────────
# Build a TensorRT INT8 engine using 500-image COCO val2017 calibration set.
# Calibration implements IInt8EntropyCalibrator2 with letterbox preprocessing
# to match the activation distribution seen at inference time.
# Calibration note: images drawn from val2017 evaluation set — INT8 mAP may be
# marginally optimistic. See data/calibration/manifest.json for details.
# Idempotent: calibration cache is written to disk; engine skipped if exists.

import cv2
import json
import pycuda.driver as cuda
import pycuda.autoinit
import tensorrt as trt

CALIB_DIR = REPO_DIR / "data/calibration"
INT8_ENGINE = ENGINE_DIR / "yolov8n_int8.engine"

# Copy calibration set from Drive if not already present
if not CALIB_DIR.exists() or not (CALIB_DIR / "manifest.json").exists():
    drive_calib = DRIVE_BASE / "calibration"
    assert drive_calib.exists(), (
        f"Calibration set not found at {CALIB_DIR} or Drive ({drive_calib}).\n"
        "Generate locally: python scripts/run_benchmark.py (creates data/calibration/)\n"
        "Then upload data/calibration/ to Drive at edge-inference-benchmark/calibration/"
    )
    shutil.copytree(str(drive_calib), str(CALIB_DIR))
    n_imgs = len(list(CALIB_DIR.glob("*.jpg")))
    print(f"Calibration set loaded from Drive: {n_imgs} images")
else:
    n_imgs = len(list(CALIB_DIR.glob("*.jpg")))
    print(f"Calibration set already present: {n_imgs} images")


class CocoInt8Calibrator(trt.IInt8EntropyCalibrator2):
    """INT8 entropy calibrator using COCO val2017 letterboxed images.

    Applies the same letterbox preprocessing used at inference time so that
    calibration activations match the deployment data distribution.
    """

    def __init__(self, calib_dir: str, batch_size: int = 1) -> None:
        super().__init__()
        calib_path = Path(calib_dir)
        manifest = json.loads((calib_path / "manifest.json").read_text())
        self._image_paths = [str(calib_path / name) for name in manifest["images"]]
        self._batch_size = batch_size
        self._index = 0
        n_bytes = batch_size * 3 * 640 * 640 * np.float32().nbytes
        self._device_input = cuda.mem_alloc(n_bytes)
        self._cache_path = ENGINE_DIR / "int8_calibration.cache"
        print(f"Calibrator: {len(self._image_paths)} images, cache at {self._cache_path}")

    def _preprocess(self, img_path: str) -> np.ndarray:
        """Letterbox-preprocess a JPEG to (1, 3, 640, 640) float32."""
        from src.data.coco_loader import letterbox_preprocess
        bgr = cv2.imread(img_path)
        if bgr is None:
            return np.zeros((1, 3, 640, 640), dtype=np.float32)
        tensor, _ = letterbox_preprocess(bgr)
        return tensor

    def get_batch_size(self) -> int:
        return self._batch_size

    def get_batch(self, names):
        if self._index >= len(self._image_paths):
            return None
        batch = self._preprocess(self._image_paths[self._index])
        cuda.memcpy_htod(self._device_input, np.ascontiguousarray(batch))
        self._index += 1
        if self._index % 100 == 0:
            print(f"  Calibrated {self._index}/{len(self._image_paths)} images...")
        return [int(self._device_input)]

    def read_calibration_cache(self):
        if self._cache_path.exists():
            print("Loading calibration cache from disk.")
            return self._cache_path.read_bytes()
        return None

    def write_calibration_cache(self, cache: bytes) -> None:
        self._cache_path.write_bytes(cache)
        print(f"Calibration cache written: {self._cache_path}")


if not INT8_ENGINE.exists():
    calibrator = CocoInt8Calibrator(str(CALIB_DIR))
    build_trt_engine(str(LOCAL_MODEL), str(INT8_ENGINE), precision="int8", calibrator=calibrator)
else:
    print(f"INT8 engine already exists ({INT8_ENGINE.stat().st_size / 1e6:.1f} MB), skipping build.")

In [ ]:
# ── Cell 6 ── Latency Benchmark ───────────────────────────────────────────────
# Run 100-run latency benchmark (10 warmup discarded) for each TRT precision.
# Uses time.perf_counter() for sub-millisecond resolution per benchmark protocol.
# Pinned host memory (pagelocked) minimises PCIe transfer overhead.
# Idempotent: runs are stateless; GPU clock state may vary between Colab sessions.

import time
import statistics
import pycuda.driver as cuda
import pycuda.autoinit
import tensorrt as trt

N_WARMUP = 10   # Discarded — populates GPU caches and compiled kernels
N_RUNS = 100    # Timed benchmark runs per benchmark protocol
INPUT_SHAPE = (1, 3, 640, 640)


class TRTSession:
    """Minimal TensorRT inference wrapper for latency benchmarking.

    Allocates pinned host buffers and device buffers once at construction.
    Inference is fully synchronous — execute_async_v2 followed by stream sync.
    """

    def __init__(self, engine_path: str) -> None:
        trt_logger = trt.Logger(trt.Logger.WARNING)
        runtime = trt.Runtime(trt_logger)
        with open(engine_path, "rb") as f:
            self._engine = runtime.deserialize_cuda_engine(f.read())
        self._context = self._engine.create_execution_context()

        n_in = int(np.prod(INPUT_SHAPE))
        n_out = 1 * 84 * 8400
        self._h_input = cuda.pagelocked_empty(n_in, dtype=np.float32)
        self._h_output = cuda.pagelocked_empty(n_out, dtype=np.float32)
        self._d_input = cuda.mem_alloc(self._h_input.nbytes)
        self._d_output = cuda.mem_alloc(self._h_output.nbytes)
        self._stream = cuda.Stream()

    def infer(self, input_tensor: np.ndarray) -> np.ndarray:
        """Single synchronous inference pass. Returns (1, 84, 8400) float32."""
        np.copyto(self._h_input, input_tensor.ravel())
        cuda.memcpy_htod_async(self._d_input, self._h_input, self._stream)
        self._context.execute_async_v2(
            bindings=[int(self._d_input), int(self._d_output)],
            stream_handle=self._stream.handle,
        )
        cuda.memcpy_dtoh_async(self._h_output, self._d_output, self._stream)
        self._stream.synchronize()
        return self._h_output.reshape(1, 84, 8400).copy()


def benchmark_latency(session: TRTSession, name: str) -> dict:
    """Profile latency: 10 warmup passes discarded, 100 timed runs measured."""
    dummy = np.zeros(INPUT_SHAPE, dtype=np.float32)

    for _ in range(N_WARMUP):
        session.infer(dummy)

    latencies = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        session.infer(dummy)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)

    result = {
        "runtime": name,
        "mean_ms": statistics.mean(latencies),
        "stddev_ms": statistics.stdev(latencies),
        "p95_ms": float(np.percentile(latencies, 95)),
        "min_ms": min(latencies),
        "max_ms": max(latencies),
        "n_runs": N_RUNS,
        "n_warmup": N_WARMUP,
    }
    print(
        f"{name:25s}  mean={result['mean_ms']:.2f}ms  "
        f"p95={result['p95_ms']:.2f}ms  stddev={result['stddev_ms']:.2f}ms  "
        f"fps={1000/result['mean_ms']:.1f}"
    )
    return result


ENGINES = {
    "tensorrt_fp32": str(FP32_ENGINE),
    "tensorrt_fp16": str(FP16_ENGINE),
    "tensorrt_int8": str(INT8_ENGINE),
}

latency_results = {}
print(f"{'Runtime':25s}  {'mean':>10}  {'p95':>10}  {'stddev':>10}  {'fps':>8}")
print("-" * 75)
for name, engine_path in ENGINES.items():
    session = TRTSession(engine_path)
    latency_results[name] = benchmark_latency(session, name)
    del session  # release GPU memory between sessions

print(f"\nBenchmark complete: {N_RUNS} runs per precision ({N_WARMUP} warmup discarded).")

In [ ]:
# ── Cell 7 ── Accuracy Evaluation (mAP@0.5:0.95) ─────────────────────────────
# Evaluate each TRT precision variant against the full COCO val2017 set (5000 images).
# mAP delta is computed relative to TRT FP32 baseline — never cross-runtime.
# Requires COCO val2017 images + annotations on Drive or already present.
# Idempotent: COCOeval is stateless per call.

from src.benchmark.accuracy_evaluator import evaluate_map, compute_map_delta, AccuracyResult
from src.data.coco_loader import CocoLoader

COCO_DIR = REPO_DIR / "data/val2017"
ANNOTATIONS = REPO_DIR / "data/annotations/instances_val2017.json"

# Copy COCO data from Drive if not present
if not COCO_DIR.exists() or not ANNOTATIONS.exists():
    drive_val = DRIVE_BASE / "val2017"
    drive_ann = DRIVE_BASE / "annotations"
    assert drive_val.exists() and drive_ann.exists(), (
        f"COCO val2017 not found at {COCO_DIR} or Drive ({drive_val}).\n"
        "Upload data/val2017/ to Drive at edge-inference-benchmark/val2017/\n"
        "Upload data/annotations/ to Drive at edge-inference-benchmark/annotations/"
    )
    shutil.copytree(str(drive_val), str(COCO_DIR))
    shutil.copytree(str(drive_ann), str(REPO_DIR / "data/annotations"))
    print(f"COCO val2017 loaded from Drive: {len(list(COCO_DIR.glob('*.jpg')))} images")

loader = CocoLoader(images_dir=str(COCO_DIR), annotations_file=str(ANNOTATIONS))
print(f"COCO val2017: {len(loader)} images")


class TRTRuntimeAdapter:
    """Wraps TRTSession to satisfy the BaseRuntime.infer interface expected by evaluate_map."""

    def __init__(self, session: TRTSession, name: str) -> None:
        self._session = session
        self.name = name

    def infer(self, input_tensor: np.ndarray) -> np.ndarray:
        return self._session.infer(input_tensor)


accuracy_results = {}
fp32_baseline = None

for name, engine_path in ENGINES.items():
    print(f"\nEvaluating mAP for {name}...")
    session = TRTSession(engine_path)
    adapter = TRTRuntimeAdapter(session, name)
    result = evaluate_map(adapter, loader, str(ANNOTATIONS), conf_threshold=0.5)
    accuracy_results[name] = result
    if name == "tensorrt_fp32":
        fp32_baseline = result
    del session

print("\n=== mAP Results ===")
print(f"{'Runtime':25s}  {'mAP@50:95':>10}  {'mAP@50':>8}  {'delta vs FP32':>14}")
print("-" * 65)
for name, result in accuracy_results.items():
    delta = (
        0.0 if name == "tensorrt_fp32"
        else compute_map_delta(fp32_baseline, result)
    )
    print(f"{name:25s}  {result.map_50_95:>10.4f}  {result.map_50:>8.4f}  {delta:>+14.4f}")

In [ ]:
# ── Cell 8 ── Export Results ───────────────────────────────────────────────────
# Assemble BenchmarkResult records, write JSON (one per precision) and summary CSV.
# Copies results back to Google Drive for persistent storage and repo integration.
# Idempotent: result files are overwritten on each run with deterministic content.

from src.results.result_schema import BenchmarkResult
from src.results.result_writer import ResultWriter
from src.utils.device_info import get_device_info

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

device_info = get_device_info()
device_info["gpu"] = torch.cuda.get_device_name(0)
device_info["gpu_memory_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
device_info["colab_env"] = colab_env

writer = ResultWriter(output_dir=str(RESULTS_DIR))
all_results = []

precision_map = {"tensorrt_fp32": "fp32", "tensorrt_fp16": "fp16", "tensorrt_int8": "int8"}

for name, lat in latency_results.items():
    precision = precision_map[name]
    acc = accuracy_results.get(
        name,
        AccuracyResult(map_50_95=0.0, map_50=0.0, precision=precision, runtime=name),
    )
    fp32_acc = accuracy_results.get("tensorrt_fp32")
    delta = (
        0.0 if precision == "fp32"
        else compute_map_delta(fp32_acc, acc) if fp32_acc else float("nan")
    )
    # GPU peak memory: max allocated since last reset, in MB
    torch.cuda.reset_peak_memory_stats()
    peak_mb = torch.cuda.max_memory_allocated() / 1e6

    result = BenchmarkResult(
        runtime=name,
        precision=precision,
        hardware="colab_t4",
        mean_latency_ms=lat["mean_ms"],
        stddev_latency_ms=lat["stddev_ms"],
        p95_latency_ms=lat["p95_ms"],
        min_latency_ms=lat["min_ms"],
        max_latency_ms=lat["max_ms"],
        fps=1000.0 / lat["mean_ms"],
        map_50_95=acc.map_50_95,
        map_50=acc.map_50,
        map_delta_vs_fp32=delta,
        peak_memory_mb=peak_mb,
        n_runs=lat["n_runs"],
        n_warmup=lat["n_warmup"],
        onnxruntime_version=device_info.get("onnxruntime_version", "N/A"),
        torch_version=torch.__version__,
        hardware_info=device_info,
    )
    writer.write_json(result)
    all_results.append(result)
    print(f"Written: results/{name}.json")

writer.write_csv(all_results)
print(f"Written: results/summary.csv ({len(all_results)} rows)")

# Back up results to Drive for persistent storage
drive_results = DRIVE_BASE / "results"
drive_results.mkdir(parents=True, exist_ok=True)
for f in RESULTS_DIR.glob("*.json"):
    shutil.copy(f, drive_results / f.name)
csv_path = RESULTS_DIR / "summary.csv"
if csv_path.exists():
    shutil.copy(csv_path, drive_results / "summary.csv")
print(f"\nResults backed up to Drive: {drive_results}")
print(f"\n=== TensorRT Benchmark Complete ===")
print(f"FP32: {latency_results['tensorrt_fp32']['mean_ms']:.2f}ms  "
      f"FP16: {latency_results['tensorrt_fp16']['mean_ms']:.2f}ms  "
      f"INT8: {latency_results['tensorrt_int8']['mean_ms']:.2f}ms")